In [ ]:
import pandas as pd
import numpy as np
import os
import datetime as dt
import missingno as msno
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import plotly.express as px
from skimpy import skim
import scipy.stats as stats

### Data frame creator

In [ ]:
df_polish = pd.read_csv('../data/smog_polishraw.csv', encoding='utf-8-sig')
#df_smog = pd.read_csv('../data/smog_raw.csv', encoding='utf-8-sig')
df_smog = pd.read_csv('../data/smog_polishraw.csv', encoding='utf-8-sig')
df_smog2 = pd.read_csv('../data/smog_raw2.csv', encoding='utf-8-sig')

### Pokazanie czy działa formatowanie - brak błędów polskich znaków - niestety nie potrafie naprawić

In [ ]:
print(df_smog['NAME'].head())

### Nazwy nagłówków

In [ ]:
print(df_smog.columns)

### Opis danych [wszystkie dane wypisane jako bool]

In [ ]:
df_smog.isna().info()

### Opis typu danych w kolumnach

In [ ]:
print(df_smog.dtypes)

### Zmiana typu danych w kolumnie 'Date' na datetime

In [ ]:
df_smog['TIMESTAMP_DATETIME'] = pd.to_datetime(df_smog['TIMESTAMP'], errors='coerce')

### Wyciągam datę

In [ ]:
df_smog['TYLKO_DATA'] = df_smog['TIMESTAMP_DATETIME'].dt.date

### Wyciągamy sam czas (Format: GG:MM:SS)

In [ ]:
df_smog['TYLKO_CZAS'] = df_smog['TIMESTAMP_DATETIME'].dt.time

### Usuwamy kolumnę pomocniczą

In [ ]:
df_smog = df_smog.drop(columns=['TIMESTAMP_DATETIME'])

### Podgląd danych po zmianie formatowania daty i czasu

In [ ]:
print(df_smog['TYLKO_CZAS'].head())

### Podgląd nazw nagłówków w tabeli

In [ ]:
df_smog.head(2)

### Podstawowe statystyki opisowe [według typów danych]

In [ ]:
df_smog.describe()

Statystyki

In [ ]:
display(df_smog.describe())

Pokaż braki

In [ ]:
print(df_smog.isnull().sum())

Pokaż dane temperatury

In [ ]:
df_smog.sort_values(['TEMPERATURE_AVG']).head(10)

Czyszczenie błędów w danych - temperatura powinna byc powyżej >10 stopni celc [ do naprawy]

In [ ]:
#df_smog = df_smog[df_smog['TEMPERATURE_AVG'] >= 10]

Usuwanie wierszy z brakami w lokalizacjach

In [ ]:
df_smog = df_smog[(df_smog['LATITUDE'] != 0) & (df_smog['LONGITUDE'] != 0)]

### Mapa braków

In [ ]:
sns.heatmap(df_smog.isna())

Przyczyny braku

In [ ]:
msno.matrix(df_smog)

Przyczyny braków

In [ ]:
msno.heatmap(df_smog)

### Set variables

In [ ]:
area_dict = {
    range(0, 10): 'Mazowieckie',
    range(10, 15): 'Warmińsko-mazurskie',
    range(15, 20): 'Podlaskie',
    range(20, 25): 'Lubelskie',
    range(25, 29): 'Świętokrzyskie',
    range(30, 35): 'Małopolskie',
    range(35, 40): 'Podkarpackie',
    range(40, 45): 'Śląskie',
    range(45, 50): 'Opolskie',
    range(50, 60): 'Dolnośląskie',
    range(60, 65): 'Wielkopolskie',
    range(66, 70): 'Lubuskie',
    range(70, 79): 'Zachodniopomorskie',
    range(80, 85): 'Pomorskie',
    range(86, 90): 'Kujawsko-pomorskie',
    range(90, 98): 'Łódzkie'
}
smog_columns = ['LONGITUDE', 'LATITUDE', 'HUMIDITY_AVG', 'PRESSURE_AVG', 'TEMPERATURE_AVG', 'PM10_AVG', 'PM25_AVG']

In [ ]:
#for x in smog_columns:
    #df_smog[f'{x}'] = pd.Series(df_smog[f'{x}'], dtype=np.dtype("float"))

# Insert area column based on post code

In [ ]:
def set_area(area_value):
    for x, y in area_dict.items():
        if area_value in x:
            area_value = y
    return area_value
area = []
for x, y in df_smog['POST_CODE'].astype(str).str[:3].astype(int).items():
    area.append(set_area(y))
df_smog.insert(2, 'AREA', area)

Dendogram

In [ ]:
msno.dendrogram(df_smog)

### Format float values with precission to 1 [ Convert values to float]

In [ ]:
'''for x in smog_columns:
    for z, y in df_smog[f"{x}"].items():
      y = float(y)
    df_smog[f"{x}"] = df_smog[f"{x}"].round(1)'''

### Strip '-' from post code

In [ ]:
for x in df_smog['POST_CODE']:
    df_smog['POST_CODE'] = df_smog['POST_CODE'].str.replace("-", "")
    df_smog2['POST_CODE'] = df_smog2['POST_CODE'].str.replace("-", "")

# Sort values by area

In [ ]:
df_smog = df_smog.sort_values(['POST_CODE'])

### Check, if every area is in place (longitude and latitude are in Polish area)

In [ ]:
# Define a function and area latitude and longitude min and max values
for y, x in df_smog['LATITUDE'].items() and df_smog['LONGITUDE'].items():
    x = float(x)

def check_areas(df):
    area_values = {
        "Latitude.North < 54.835563": df_smog['LATITUDE'] < 54.8,
        "Latitude.South > 49.002063": df_smog['LATITUDE'] > 49.0,
        "Longitude.East < 24.145562": df_smog['LONGITUDE'] < 24.2,
        "Longitude.West > 14.124562": df_smog['LONGITUDE'] > 14.0
        }
    return area_values

# Define an object table with the area checker

areas = check_areas(df_smog)

# Print if there are any differences in the areas

for rule, result in areas.items():
    print(f"{rule}: {not result.all()}")

### Check, how many of areas have wrong latitudes\longitudes

In [ ]:
# Check, how many differences are in areas

differences = {rule: ~result for rule, result in areas.items()}
summary = {rule: result.sum() for rule, result in differences.items()}

# Print the number of differences

for rule, count in summary.items():
    print(f"{rule}: {count} differences")

Szukanie korelacji

In [ ]:
sns.barplot(data=df_smog,x='POST_CODE',y='PM10_AVG',errorbar=None,estimator=np.mean,color='steelblue')

In [ ]:
sns.barplot(data=df_smog,x='TEMPERATURE_AVG',y='PM10_AVG',errorbar=None,estimator=np.mean,color='steelblue')

In [ ]:
korelacje = df_smog.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(10, 8))

In [ ]:
sns.heatmap(korelacje, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)

Usuwanie duplikatów

In [ ]:
korelacje_lista = korelacje.unstack().sort_values(key=abs, ascending=False).drop_duplicates()

In [ ]:
korelacje_lista = korelacje_lista[korelacje_lista != 1.0]

In [ ]:
print("Najsilniejsze znalezione korelacje w danych:")

In [ ]:
print(korelacje_lista.head(10))

Histogramy dla PM10

In [ ]:
p=plt.hist(df_smog['PM10_AVG'], bins=30, color='steelblue', edgecolor='black')
plt.ylabel('Liczba pomiarów')
plt.title('Histogram PM10')
plt.xlabel('Wartość PM10')

Wykres rozrzutu

In [ ]:
p=plt.scatter(df_smog['PM10_AVG'], df_smog['PM25_AVG'], color='steelblue', edgecolor='black')
plt.xlabel('Wartość PM10')
plt.ylabel('Wartość PM2.5')
plt.title('Wykres rozrzutu PM10 vs PM2.5')

Wykres rozrzutu

In [ ]:
sns.histplot(data=df_smog, x='PM10_AVG',stat='probability', bins=50, color='steelblue', edgecolor='black')

Wykres rozrzutu

In [ ]:
sns.catplot(data=df_smog,x='LATITUDE', y='PM10_AVG',kind='point')

Analiza opisowa

In [ ]:
sns.scatterplot(data=df_smog, x='TEMPERATURE_AVG', y='PM25_AVG')

Rozrzut PM10 wg temperatury

In [ ]:
sns.scatterplot(data=df_smog, x='TEMPERATURE_AVG', y='PM10_AVG')

Analiza opisowa

In [ ]:
skim(df_smog)

Zliczamy ilośc pomiarów dla każdej wartości PM10 w danych:

In [ ]:
df_smog['PM10_AVG'].value_counts()

Definiujemy prostą funkcję do obliczenia współczynnika zmienności (w procentach)


In [ ]:
def wsp_zmiennosci(x):
    return (x.std() / x.mean()) * 100

Wywołujemy interesujące nas statystyki dla konkretnej kolumny (lub kilku kolumn)

In [ ]:
statystyki = df_smog[['PM10_AVG']].agg([
    'mean',                   # Średnia
    'median',                 # Mediana (Q2)
    'std',                    # Odchylenie standardowe
    wsp_zmiennosci,           # Współczynnik zmienności (nasza funkcja)
    lambda x: x.quantile(0.25), # Q1
    lambda x: x.quantile(0.75)  # Q3
    ])

Najwyższe zanieczyszczenie

In [ ]:
top_5 = df_smog.nlargest(5, 'PM10_AVG')
display(top_5)

Najnizsze zanieczyszczenie

In [ ]:
bottom_5 = df_smog.nsmallest(5, 'PM10_AVG')
display(bottom_5)

In [ ]:
print(statystyki)

Wyniki:
Średnia (mean) wynosi ok. 3.62, podczas gdy mediana (median) to zaledwie 2.02.
Interpretacja: Mediana mówi nam, że w dokładnie połowie badanych przypadków zanieczyszczenie pyłem PM10 wynosiło zaledwie 2.02 lub mniej. Dobowa norma alarmowa to 50. Ponieważ średnia jest zauważalnie wyższa od mediany, mamy tu do czynienia z rozkładem silnie prawoskośnym. Oznacza to, że przez większość czasu powietrze jest bardzo czyste, ale występują nagłe, rzadkie godziny z gigantycznym smogiem, które matematycznie "ciągną" średnią w górę.

Odchylenie standardowe (std) wynosi 6.86, a współczynnik zmienności osiągnął 189%.
Interpretacja: Współczynnik zmienności powyżej 100% oznacza ekstremalnie wysoki rozrzut danych. Odchylenie standardowe jest prawie dwukrotnie wyższe od samej średniej. Zjawisko smogu jest niestabilne i epizodyczne.

Dolny kwartyl (Q1) to 1.52, a górny kwartyl (Q3) to 2.75.
Interpretacja: Te liczby oznaczają, że aż połowa (środkowe 50%) wszystkich zebranych pomiarów jest w przedziale między 1.52 a 2.75.Q3 wynosi 2.75, co oznacza, że aż 75% wszystkich pomiarów jest niższych niż 2.75.Problemy ze smogiem (i za podwyższoną średnią oraz ogromne odchylenie) odpowiada najwyższe 25% pomiarów.

### Save dataframe to csv file **Always put as a last cell !**

In [ ]:
ts = dt.datetime.now().strftime("%Y%m%d_%H-%M")
#df_smog.to_csv(f"../tests/smog_raw{ts}.csv", index=False)
#df_smog2.to_csv(f"../tests/smog2_raw{ts}.csv", index=False)
#df_polish.to_csv(f"../tests/smog_polishraw{ts}.csv", index=False, encoding='utf-8-sig')